# Chapter 13: Debugging Workflows That Didn't Fire (Reference)

## Learning Objectives

- Diagnose a YAML syntax error and report its line
- Replay paths: filter matching against a changed-file list
- Check whether a fired event type is declared under on:
- Recite the ordered diagnostic checklist

## Setup

The next cell sets up reproducibility and the `PRA_MODE` toggle. You should see `PRA_MODE = 'fixture'` printed by default.

In [ ]:
import os
import random
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pr_automerge").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

RANDOM_STATE: int = 42
random.seed(RANDOM_STATE)

PRA_MODE = os.environ.get("PRA_MODE", "fixture")
PRA_REPO = os.environ.get("PRA_REPO", "")

if PRA_MODE == "live":
    assert PRA_REPO, "Set PRA_REPO=owner/name to run against a real repo"

print(f"PRA_MODE = {PRA_MODE!r}")

## 1. YAML Syntax Diagnosis

The next cell checks a deliberately broken YAML snippet, then a well-formed one. You should see `valid=False` with a line number for the first, `valid=True` for the second.

In [ ]:
from labs.lab_13_why_no_trigger import diagnose_yaml_syntax

broken = "on:\n  pull_request:\n  paths:\n   - 'sandbox/**'\n  bad indent here"
result = diagnose_yaml_syntax(broken)
print(f"valid={result['valid']}, line={result['line']}, error={result['error']}")

good = "on:\n  pull_request:\n    paths:\n      - 'sandbox/**'\n"
print(f"valid={diagnose_yaml_syntax(good)['valid']} (well-formed example)")

## 2. Path-Filter Replay

The next cell replays `paths: ['sandbox/**']` against two different changed-file sets. You should see the curriculum-only PR correctly NOT match, and the sandbox PR correctly match.

In [ ]:
from labs.lab_13_why_no_trigger import check_path_filter_match

curriculum_pr = ["learning_modules/chapter_13_debugging_workflows.md"]
sandbox_pr = ["sandbox/generated/pr_120.py"]
print(
    f"curriculum PR matches sandbox/**? {check_path_filter_match(curriculum_pr, ['sandbox/**'])}"
)
print(
    f"sandbox PR matches sandbox/**?     {check_path_filter_match(sandbox_pr, ['sandbox/**'])}"
)

## 3. Event-Type Matching

The next cell checks three candidate fired events against a sample `on:` block. You should see `pull_request` and `workflow_dispatch` match, `push` not match.

In [ ]:
from labs.lab_13_why_no_trigger import check_event_type_match

on_block = {"pull_request": {"paths": ["sandbox/**"]}, "workflow_dispatch": {}}
for event in ["pull_request", "push", "workflow_dispatch"]:
    print(
        f"fired={event!r} matches on: block? {check_event_type_match(event, on_block)}"
    )

## 4. Reading the Engine's Own Trace (`ACTIONS_STEP_DEBUG`)

Checks 1–3 above get a run to *exist*. When a run exists but a step silently skips, set a repository secret or variable `ACTIONS_STEP_DEBUG=true` and the engine narrates its own expression evaluation, verbatim (captured from this repo's Gate 2 run — Chapter 13 §9):

```
##[debug]Evaluating condition for step: 'Evaluate Gate 2 and publish a check run'
##[debug]Evaluating: success()
##[debug]Evaluating success:
##[debug]=> true
##[debug]Result: true
##[debug]Starting: Evaluate Gate 2 and publish a check run
```

**What to notice:** the `##[debug]Evaluating:` line shows the exact `if:` expression and `Result:` its verdict — when a step you expected to run was skipped, this is the line that says why. No code cell here: this trace only comes from a real run's log (`ACTIONS_RUNNER_DEBUG=true` additionally adds the downloadable runner-diagnostic archive).


## Takeaways & Next Steps

This notebook's takeaway is the path-filter replay above -- a curriculum-only PR correctly triggering zero gates is this repo's design working as intended, not a bug.

In [ ]:
from labs.lab_13_why_no_trigger import DIAGNOSTIC_CHECKLIST

for i, item in enumerate(DIAGNOSTIC_CHECKLIST, start=1):
    print(f"[{i}] {item}")

---

📖 **Reading companion:** [Chapter 13: Debugging Workflows That Didn't Fire](../learning_modules/chapter_13_debugging_workflows.md)
🔬 **Try it live:** this chapter's lab is fully offline (no `gh` calls), so `PRA_MODE=live` changes nothing here — nothing to re-run against a real repo.
